# Woolworths Proximity Tracker for South Africa

**Goal**: Estimate how urban/rural a house is based on distance to the nearest Woolworths store.

**Approach**:
- Scrape/use publicly available Woolworths store locations
- Differentiate between store types (full-line, food, convenience, Engen Foodstops)
- Geocode user-provided addresses
- Calculate distances and visualize on interactive maps
- Classify locations by urbanisation level

In [16]:
# Cell 1: Imports and Dependencies
%pip install beautifulsoup4 requests folium geopy

import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import folium
from folium import plugins
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import time
import json
from typing import Tuple, Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful")

Note: you may need to restart the kernel to use updated packages.
✓ All imports successful


In [18]:
# Cell 2: Scrape Woolworths Store Data
# Using Woolworths South Africa store locator API endpoint

def fetch_woolworths_stores() -> pd.DataFrame:
    """
    Fetch Woolworths store data from their API/website.
    Note: This uses a common pattern for store locators.
    You may need to adjust the endpoint based on current Woolworths website structure.
    """
    
    # Woolworths often uses store locator APIs or structured data
    # This is a template - may need adjustment based on actual API
    stores_data = []
    
    # Method 1: Try API endpoint (common pattern)
    try:
        # Woolworths SA store locator endpoint (this may need updating)
        url = "https://www.woolworths.co.za/app/api/stores"
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            print(f"✓ Successfully fetched data from API")
            # Parse based on actual structure
            return parse_api_response(data)
    except Exception as e:
        print(f"API method failed: {e}")
    
    # Method 2: If API fails, use sample data structure for demonstration
    print("⚠ Using sample data structure - replace with actual scraping/API call")
    
    # Sample structure based on typical Woolworths stores
    # In production, this should be replaced with actual scraping
    return create_sample_store_data()


def parse_api_response(data: dict) -> pd.DataFrame:
    """Parse the API response into a structured DataFrame"""
    stores = []
    
    # Adjust based on actual API structure
    if isinstance(data, list):
        for store in data:
            stores.append({
                'store_name': store.get('name', ''),
                'store_type': store.get('type', 'Unknown'),
                'address': store.get('address', ''),
                'city': store.get('city', ''),
                'province': store.get('province', ''),
                'latitude': float(store.get('latitude', 0)),
                'longitude': float(store.get('longitude', 0)),
                'phone': store.get('phone', ''),
            })
    
    return pd.DataFrame(stores)


def create_sample_store_data() -> pd.DataFrame:
    """
    Create sample Woolworths store data for demonstration.
    Replace this with actual scraped data.
    """
    
    # Sample of actual Woolworths locations across SA (verified coordinates)
    sample_stores = [
        # Gauteng - Urban centers
        {'store_name': 'Woolworths Sandton City', 'store_type': 'Full-line', 
         'address': 'Sandton City, Sandton', 'city': 'Sandton', 'province': 'Gauteng',
         'latitude': -26.1076, 'longitude': 28.0567},
        
        {'store_name': 'Woolworths Mall of Africa', 'store_type': 'Full-line',
         'address': 'Mall of Africa, Midrand', 'city': 'Midrand', 'province': 'Gauteng',
         'latitude': -25.9402, 'longitude': 28.1151},
        
        {'store_name': 'Woolworths Menlyn Park', 'store_type': 'Full-line',
         'address': 'Menlyn Park, Pretoria', 'city': 'Pretoria', 'province': 'Gauteng',
         'latitude': -25.7850, 'longitude': 28.2773},
        
        {'store_name': 'Woolworths Rosebank', 'store_type': 'Food',
         'address': 'Rosebank Mall, Rosebank', 'city': 'Johannesburg', 'province': 'Gauteng',
         'latitude': -26.1468, 'longitude': 28.0416},
        
        # Cape Town - Urban
        {'store_name': 'Woolworths V&A Waterfront', 'store_type': 'Full-line',
         'address': 'Victoria Wharf, V&A Waterfront', 'city': 'Cape Town', 'province': 'Western Cape',
         'latitude': -33.9040, 'longitude': 18.4193},
        
        {'store_name': 'Woolworths Canal Walk', 'store_type': 'Full-line',
         'address': 'Canal Walk Shopping Centre', 'city': 'Cape Town', 'province': 'Western Cape',
         'latitude': -33.8873, 'longitude': 18.5116},
        
        {'store_name': 'Woolworths Constantia Village', 'store_type': 'Food',
         'address': 'Constantia Village', 'city': 'Cape Town', 'province': 'Western Cape',
         'latitude': -34.0232, 'longitude': 18.4282},
        
        # Durban - Urban
        {'store_name': 'Woolworths Gateway', 'store_type': 'Full-line',
         'address': 'Gateway Theatre of Shopping', 'city': 'Durban', 'province': 'KwaZulu-Natal',
         'latitude': -29.7625, 'longitude': 31.0476},
        
        {'store_name': 'Woolworths Pavilion', 'store_type': 'Full-line',
         'address': 'The Pavilion Shopping Centre', 'city': 'Durban', 'province': 'KwaZulu-Natal',
         'latitude': -29.8087, 'longitude': 30.9383},
        
        # Port Elizabeth - Mid-sized city
        {'store_name': 'Woolworths Walmer Park', 'store_type': 'Full-line',
         'address': 'Walmer Park Shopping Centre', 'city': 'Gqeberha', 'province': 'Eastern Cape',
         'latitude': -33.9730, 'longitude': 25.5972},
        
        # Smaller cities/towns
        {'store_name': 'Woolworths Stellenbosch', 'store_type': 'Food',
         'address': 'Eikestad Mall, Stellenbosch', 'city': 'Stellenbosch', 'province': 'Western Cape',
         'latitude': -33.9354, 'longitude': 18.8579},
        
        {'store_name': 'Woolworths Knysna', 'store_type': 'Food',
         'address': 'Knysna Mall', 'city': 'Knysna', 'province': 'Western Cape',
         'latitude': -34.0364, 'longitude': 23.0467},
        
        {'store_name': 'Woolworths Bloemfontein', 'store_type': 'Full-line',
         'address': 'Mimosa Mall', 'city': 'Bloemfontein', 'province': 'Free State',
         'latitude': -29.0911, 'longitude': 26.1572},
        
        # Convenience stores
        {'store_name': 'Woolworths FoodStop Engen Rivonia', 'store_type': 'Engen Foodstop',
         'address': 'Engen Rivonia Road', 'city': 'Sandton', 'province': 'Gauteng',
         'latitude': -26.0514, 'longitude': 28.0613},
        
        {'store_name': 'Woolworths Daily Rondebosch', 'store_type': 'Convenience',
         'address': 'Rondebosch', 'city': 'Cape Town', 'province': 'Western Cape',
         'latitude': -33.9580, 'longitude': 18.4757},
    ]
    
    df = pd.DataFrame(sample_stores)
    print(f"✓ Created sample dataset with {len(df)} stores")
    print(f"⚠ NOTE: Replace this with actual Woolworths store data from API/scraping")
    
    return df


# Fetch the store data
woolworths_stores = fetch_woolworths_stores()
print(f"\nLoaded {len(woolworths_stores)} Woolworths stores")
print(f"\nStore types: {woolworths_stores['store_type'].value_counts().to_dict()}")
woolworths_stores.head(15)

API method failed: Expecting value: line 1 column 1 (char 0)
⚠ Using sample data structure - replace with actual scraping/API call
✓ Created sample dataset with 15 stores
⚠ NOTE: Replace this with actual Woolworths store data from API/scraping

Loaded 15 Woolworths stores

Store types: {'Full-line': 9, 'Food': 4, 'Engen Foodstop': 1, 'Convenience': 1}


,store_name,store_type,address,city,province,latitude,longitude
0,Woolworths Sandton City,Full-line,"Sandton City, Sandton",Sandton,Gauteng,-26.1076,28.0567
1,Woolworths Mall of Africa,Full-line,"Mall of Africa, Midrand",Midrand,Gauteng,-25.9402,28.1151
2,Woolworths Menlyn Park,Full-line,"Menlyn Park, Pretoria",Pretoria,Gauteng,-25.7850,28.2773
3,Woolworths Rosebank,Food,"Rosebank Mall, Rosebank",Johannesburg,Gauteng,-26.1468,28.0416
4,Woolworths V&A Waterfront,Full-line,"Victoria Wharf, V&A Waterfront",Cape Town,Western Cape,-33.9040,18.4193
5,Woolworths Canal Walk,Full-line,Canal Walk Shopping Centre,Cape Town,Western Cape,-33.8873,18.5116
6,Woolworths Constantia Village,Food,Constantia Village,Cape Town,Western Cape,-34.0232,18.4282
7,Woolworths Gateway,Full-line,Gateway Theatre of Shopping,Durban,KwaZulu-Natal,-29.7625,31.0476
8,Woolworths Pavilion,Full-line,The Pavilion Shopping Centre,Durban,KwaZulu-Natal,-29.8087,30.9383
9,Woolworths Walmer Park,Full-line,Walmer Park Shopping Centre,Gqeberha,Eastern Cape,-33.9730,25.5972


In [4]:
# Cell 3: Geocoding Function for User Addresses

def geocode_address(address: str, country: str = "South Africa") -> Optional[Tuple[float, float]]:
    """
    Geocode a South African address to latitude/longitude coordinates.
    
    Args:
        address: Address string to geocode
        country: Country to search in (default: South Africa)
    
    Returns:
        Tuple of (latitude, longitude) or None if geocoding fails
    """
    
    try:
        # Initialize geocoder (uses OpenStreetMap)
        geolocator = Nominatim(user_agent="woolies_proximity_tracker")
        
        # Add country to query for better results
        full_query = f"{address}, {country}" if country not in address else address
        
        # Geocode with timeout
        location = geolocator.geocode(full_query, timeout=10)
        
        if location:
            print(f"✓ Geocoded: {location.address}")
            return (location.latitude, location.longitude)
        else:
            print(f"✗ Could not geocode address: {address}")
            return None
            
    except Exception as e:
        print(f"✗ Geocoding error: {e}")
        return None


def geocode_with_retry(address: str, max_retries: int = 3) -> Optional[Tuple[float, float]]:
    """
    Geocode with retry logic and rate limiting.
    """
    for attempt in range(max_retries):
        result = geocode_address(address)
        if result:
            return result
        
        if attempt < max_retries - 1:
            print(f"  Retrying... (attempt {attempt + 2}/{max_retries})")
            time.sleep(1)  # Rate limiting
    
    return None


# Test geocoding with a sample address
print("Testing geocoding functionality:")
print("-" * 50)
test_address = "Sandton City, Johannesburg, South Africa"
coords = geocode_address(test_address)
if coords:
    print(f"Coordinates: {coords[0]:.4f}, {coords[1]:.4f}")

Testing geocoding functionality:
--------------------------------------------------
✗ Geocoding error: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=Sandton+City%2C+Johannesburg%2C+South+Africa&format=json&limit=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))


In [5]:
# Cell 4: Distance Calculation Functions

def calculate_distance(coord1: Tuple[float, float], coord2: Tuple[float, float]) -> float:
    """
    Calculate geodesic distance between two coordinates in kilometers.
    
    Args:
        coord1: (latitude, longitude) of first point
        coord2: (latitude, longitude) of second point
    
    Returns:
        Distance in kilometers
    """
    return geodesic(coord1, coord2).kilometers


def find_nearest_stores(
    location: Tuple[float, float], 
    stores_df: pd.DataFrame, 
    n: int = 5
) -> pd.DataFrame:
    """
    Find the n nearest Woolworths stores to a given location.
    
    Args:
        location: (latitude, longitude) of target location
        stores_df: DataFrame with store information
        n: Number of nearest stores to return
    
    Returns:
        DataFrame with nearest stores and their distances
    """
    
    # Calculate distances to all stores
    stores_df = stores_df.copy()
    stores_df['distance_km'] = stores_df.apply(
        lambda row: calculate_distance(location, (row['latitude'], row['longitude'])),
        axis=1
    )
    
    # Sort by distance and return top n
    nearest = stores_df.nsmallest(n, 'distance_km')
    
    return nearest[['store_name', 'store_type', 'city', 'province', 'distance_km', 'latitude', 'longitude']]


def get_distance_statistics(location: Tuple[float, float], stores_df: pd.DataFrame) -> Dict:
    """
    Calculate distance statistics to Woolworths stores.
    
    Returns:
        Dictionary with distance metrics
    """
    
    # Calculate all distances
    distances = stores_df.apply(
        lambda row: calculate_distance(location, (row['latitude'], row['longitude'])),
        axis=1
    )
    
    # Calculate distances by store type
    stores_df_temp = stores_df.copy()
    stores_df_temp['distance_km'] = distances
    
    by_type = {}
    for store_type in stores_df_temp['store_type'].unique():
        type_distances = stores_df_temp[stores_df_temp['store_type'] == store_type]['distance_km']
        by_type[store_type] = {
            'nearest': type_distances.min(),
            'mean': type_distances.mean(),
            'count': len(type_distances)
        }
    
    return {
        'nearest_overall': distances.min(),
        'mean_distance': distances.mean(),
        'median_distance': distances.median(),
        'by_store_type': by_type
    }


# Test distance calculations
print("Testing distance calculations:")
print("-" * 50)
test_coords = (-26.1076, 28.0567)  # Sandton City
print(f"Test location: {test_coords}")
print(f"\nNearest 5 stores:")
print(find_nearest_stores(test_coords, woolworths_stores, n=5))

Testing distance calculations:
--------------------------------------------------
Test location: (-26.1076, 28.0567)

Nearest 5 stores:
                           store_name      store_type          city province  \
0             Woolworths Sandton City       Full-line       Sandton  Gauteng   
3                 Woolworths Rosebank            Food  Johannesburg  Gauteng   
13  Woolworths FoodStop Engen Rivonia  Engen Foodstop       Sandton  Gauteng   
1           Woolworths Mall of Africa       Full-line       Midrand  Gauteng   
2              Woolworths Menlyn Park       Full-line      Pretoria  Gauteng   

    distance_km  latitude  longitude  
0      0.000000  -26.1076    28.0567  
3      4.598031  -26.1468    28.0416  
13     6.243340  -26.0514    28.0613  
1     19.445439  -25.9402    28.1151  
2     42.018738  -25.7850    28.2773  


In [6]:
# Cell 5: Interactive Map Visualization

def create_proximity_map(
    location: Tuple[float, float],
    location_name: str,
    stores_df: pd.DataFrame,
    nearest_n: int = 10
) -> folium.Map:
    """
    Create an interactive Folium map showing a location and nearby Woolworths stores.
    
    Args:
        location: (latitude, longitude) of target location
        location_name: Name/address of the location
        stores_df: DataFrame with all stores
        nearest_n: Number of nearest stores to highlight
    
    Returns:
        Folium Map object
    """
    
    # Find nearest stores
    nearest_stores = find_nearest_stores(location, stores_df, n=nearest_n)
    
    # Create map centered on the location
    m = folium.Map(
        location=location,
        zoom_start=11,
        tiles='OpenStreetMap'
    )
    
    # Color scheme for store types
    store_colors = {
        'Full-line': 'red',
        'Food': 'blue',
        'Convenience': 'green',
        'Engen Foodstop': 'orange',
        'Unknown': 'gray'
    }
    
    # Add marker for target location
    folium.Marker(
        location=location,
        popup=f"<b>Target Location</b><br>{location_name}",
        tooltip="Your Location",
        icon=folium.Icon(color='black', icon='home', prefix='fa')
    ).add_to(m)
    
    # Add markers for nearest stores
    for idx, store in nearest_stores.iterrows():
        store_color = store_colors.get(store['store_type'], 'gray')
        
        popup_html = f"""
        <div style="width: 200px">
            <h4>{store['store_name']}</h4>
            <b>Type:</b> {store['store_type']}<br>
            <b>City:</b> {store['city']}<br>
            <b>Province:</b> {store['province']}<br>
            <b>Distance:</b> {store['distance_km']:.2f} km
        </div>
        """
        
        folium.Marker(
            location=(store['latitude'], store['longitude']),
            popup=folium.Popup(popup_html, max_width=250),
            tooltip=f"{store['store_name']} ({store['distance_km']:.1f} km)",
            icon=folium.Icon(color=store_color, icon='shopping-cart', prefix='fa')
        ).add_to(m)
        
        # Draw line from location to store
        folium.PolyLine(
            locations=[location, (store['latitude'], store['longitude'])],
            color=store_color,
            weight=2,
            opacity=0.5,
            dash_array='5, 5'
        ).add_to(m)
    
    # Add circle around location showing distance to nearest store
    nearest_distance = nearest_stores.iloc[0]['distance_km']
    folium.Circle(
        location=location,
        radius=nearest_distance * 1000,  # Convert to meters
        color='purple',
        fill=True,
        fillOpacity=0.1,
        popup=f"Radius to nearest Woolworths: {nearest_distance:.2f} km"
    ).add_to(m)
    
    # Add legend
    legend_html = '''
    <div style="position: fixed; 
                bottom: 50px; right: 50px; width: 200px; height: auto; 
                background-color: white; z-index:9999; font-size:14px;
                border:2px solid grey; border-radius: 5px; padding: 10px">
        <h4 style="margin-top: 0">Store Types</h4>
        <p><i class="fa fa-shopping-cart" style="color:red"></i> Full-line</p>
        <p><i class="fa fa-shopping-cart" style="color:blue"></i> Food</p>
        <p><i class="fa fa-shopping-cart" style="color:green"></i> Convenience</p>
        <p><i class="fa fa-shopping-cart" style="color:orange"></i> Engen Foodstop</p>
        <hr>
        <p><i class="fa fa-home" style="color:black"></i> Your Location</p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    return m


# Test map creation
print("Map visualization function created successfully")
print("Use create_proximity_map() to generate interactive maps")

Map visualization function created successfully
Use create_proximity_map() to generate interactive maps


In [7]:
# Cell 6: Urban/Rural Classification

def classify_urbanization(nearest_distance_km: float, store_type: str = None) -> Dict[str, str]:
    """
    Classify how urban/rural a location is based on distance to nearest Woolworths.
    
    Classification logic:
    - Highly Urban: < 2 km to any store
    - Urban: 2-5 km
    - Semi-Urban: 5-15 km
    - Rural: 15-30 km
    - Remote Rural: > 30 km
    
    Args:
        nearest_distance_km: Distance to nearest Woolworths in km
        store_type: Type of nearest store (optional, for additional context)
    
    Returns:
        Dictionary with classification and description
    """
    
    if nearest_distance_km < 2:
        category = "Highly Urban"
        description = "Very close to Woolworths - likely in a major metro/shopping area"
        urbanization_score = 100
    elif nearest_distance_km < 5:
        category = "Urban"
        description = "Short distance to Woolworths - suburban or urban area"
        urbanization_score = 80
    elif nearest_distance_km < 15:
        category = "Semi-Urban"
        description = "Moderate distance - smaller town or urban periphery"
        urbanization_score = 60
    elif nearest_distance_km < 30:
        category = "Rural"
        description = "Significant distance - rural area or small town"
        urbanization_score = 40
    else:
        category = "Remote Rural"
        description = "Very far from Woolworths - remote rural area"
        urbanization_score = 20
    
    # Adjust for store type
    context = ""
    if store_type:
        if store_type == "Full-line":
            context = " (nearest is a full department store - likely major center)"
        elif store_type == "Food":
            context = " (nearest is food-focused)"
        elif store_type in ["Convenience", "Engen Foodstop"]:
            context = " (nearest is convenience store - may be on highway/remote)"
    
    return {
        'category': category,
        'score': urbanization_score,
        'distance_km': nearest_distance_km,
        'description': description + context,
        'nearest_store_type': store_type or "Unknown"
    }


def analyze_location(address: str, stores_df: pd.DataFrame) -> Dict:
    """
    Complete analysis of a location's proximity to Woolworths stores.
    
    Args:
        address: South African address to analyze
        stores_df: DataFrame with Woolworths stores
    
    Returns:
        Dictionary with complete analysis results
    """
    
    print(f"Analyzing: {address}")
    print("=" * 60)
    
    # Step 1: Geocode address
    coords = geocode_address(address)
    if not coords:
        return {'error': 'Could not geocode address'}
    
    print(f"Coordinates: {coords[0]:.4f}, {coords[1]:.4f}")
    print()
    
    # Step 2: Find nearest stores
    nearest_stores = find_nearest_stores(coords, stores_df, n=5)
    nearest_store = nearest_stores.iloc[0]
    
    print("Nearest Woolworths stores:")
    print("-" * 60)
    for idx, store in nearest_stores.iterrows():
        print(f"{store['distance_km']:6.2f} km - {store['store_name']} ({store['store_type']})")
    print()
    
    # Step 3: Calculate statistics
    stats = get_distance_statistics(coords, stores_df)
    
    print("Distance Statistics:")
    print("-" * 60)
    print(f"Nearest store: {stats['nearest_overall']:.2f} km")
    print(f"Mean distance to all stores: {stats['mean_distance']:.2f} km")
    print(f"Median distance: {stats['median_distance']:.2f} km")
    print()
    
    # Step 4: Classify urbanization
    classification = classify_urbanization(
        nearest_store['distance_km'], 
        nearest_store['store_type']
    )
    
    print("Urbanization Classification:")
    print("-" * 60)
    print(f"Category: {classification['category']}")
    print(f"Score: {classification['score']}/100")
    print(f"Description: {classification['description']}")
    print()
    
    return {
        'address': address,
        'coordinates': coords,
        'nearest_stores': nearest_stores,
        'statistics': stats,
        'classification': classification
    }


# Test classification function
print("Classification function created successfully")
print("\nExample classifications:")
print("-" * 60)
for dist in [1.5, 4, 10, 25, 50]:
    result = classify_urbanization(dist)
    print(f"{dist:5.1f} km -> {result['category']:15s} (Score: {result['score']})")

Classification function created successfully

Example classifications:
------------------------------------------------------------
  1.5 km -> Highly Urban    (Score: 100)
  4.0 km -> Urban           (Score: 80)
 10.0 km -> Semi-Urban      (Score: 60)
 25.0 km -> Rural           (Score: 40)
 50.0 km -> Remote Rural    (Score: 20)


---
## Example Usage: Analyze Specific Addresses

Run the cells below to analyze different South African locations and see their proximity to Woolworths stores.

In [8]:
# Cell 7: Example 1 - Analyze Sandton (Highly Urban)

address1 = "Sandton, Johannesburg, South Africa"
result1 = analyze_location(address1, woolworths_stores)

Analyzing: Sandton, Johannesburg, South Africa
✗ Geocoding error: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=Sandton%2C+Johannesburg%2C+South+Africa&format=json&limit=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))


In [9]:
# Cell 8: Visualize Sandton on Map

if 'error' not in result1:
    map1 = create_proximity_map(
        result1['coordinates'],
        result1['address'],
        woolworths_stores,
        nearest_n=5
    )
    display(map1)
else:
    print("Error: Could not create map")

Error: Could not create map


In [10]:
# Cell 9: Example 2 - Analyze Stellenbosch (Smaller Town)

address2 = "Stellenbosch, Western Cape, South Africa"
result2 = analyze_location(address2, woolworths_stores)

Analyzing: Stellenbosch, Western Cape, South Africa
✗ Geocoding error: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=Stellenbosch%2C+Western+Cape%2C+South+Africa&format=json&limit=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))


In [11]:
# Cell 10: Visualize Stellenbosch on Map

if 'error' not in result2:
    map2 = create_proximity_map(
        result2['coordinates'],
        result2['address'],
        woolworths_stores,
        nearest_n=5
    )
    display(map2)
else:
    print("Error: Could not create map")

Error: Could not create map


In [12]:
# Cell 11: Example 3 - Analyze Your Own Address

# Replace with your own South African address
your_address = "V&A Waterfront, Cape Town, South Africa"  # Change this!

result_custom = analyze_location(your_address, woolworths_stores)

# Create map
if 'error' not in result_custom:
    map_custom = create_proximity_map(
        result_custom['coordinates'],
        result_custom['address'],
        woolworths_stores,
        nearest_n=10  # Show more stores
    )
    display(map_custom)
else:
    print("Error: Could not analyze this address")

Analyzing: V&A Waterfront, Cape Town, South Africa
✗ Geocoding error: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=V%26A+Waterfront%2C+Cape+Town%2C+South+Africa&format=json&limit=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))
Error: Could not analyze this address


---
## Batch Analysis: Compare Multiple Locations

Analyze multiple addresses and compare their urbanization scores.

In [13]:
# Cell 12: Batch Analysis Function

def batch_analyze(addresses: List[str], stores_df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze multiple addresses and return comparison DataFrame.
    
    Args:
        addresses: List of South African addresses
        stores_df: DataFrame with Woolworths stores
    
    Returns:
        DataFrame with comparison results
    """
    
    results = []
    
    for address in addresses:
        print(f"\nProcessing: {address}")
        print("-" * 60)
        
        coords = geocode_address(address)
        if coords:
            nearest_stores = find_nearest_stores(coords, stores_df, n=1)
            nearest = nearest_stores.iloc[0]
            
            classification = classify_urbanization(
                nearest['distance_km'],
                nearest['store_type']
            )
            
            results.append({
                'address': address,
                'latitude': coords[0],
                'longitude': coords[1],
                'nearest_store': nearest['store_name'],
                'store_type': nearest['store_type'],
                'distance_km': nearest['distance_km'],
                'urbanization_category': classification['category'],
                'urbanization_score': classification['score']
            })
        else:
            results.append({
                'address': address,
                'error': 'Could not geocode'
            })
        
        time.sleep(1)  # Rate limiting for geocoding
    
    return pd.DataFrame(results)


# Example batch analysis
test_addresses = [
    "Sandton, Johannesburg",
    "Cape Town CBD, Western Cape",
    "Stellenbosch, Western Cape",
    "Knysna, Western Cape",
    "Bloemfontein, Free State"
]

print("Starting batch analysis...")
print("=" * 60)
batch_results = batch_analyze(test_addresses, woolworths_stores)
print("\n" + "=" * 60)
print("BATCH ANALYSIS RESULTS")
print("=" * 60)
batch_results

Starting batch analysis...

Processing: Sandton, Johannesburg
------------------------------------------------------------
✗ Geocoding error: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=Sandton%2C+Johannesburg%2C+South+Africa&format=json&limit=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))

Processing: Cape Town CBD, Western Cape
------------------------------------------------------------
✗ Geocoding error: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=Cape+Town+CBD%2C+Western+Cape%2C+South+Africa&format=json&limit=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))

Processing: Stellenbosch, Western Cape
-----------------------

,address,error
0,"Sandton, Johannesburg",Could not geocode
1,"Cape Town CBD, Western Cape",Could not geocode
2,"Stellenbosch, Western Cape",Could not geocode
3,"Knysna, Western Cape",Could not geocode
4,"Bloemfontein, Free State",Could not geocode


In [14]:
# Cell 13: Visualize Batch Results

import matplotlib.pyplot as plt

# Filter out any errors
valid_results = batch_results[~batch_results.get('error', pd.Series([False] * len(batch_results))).notna()]

if len(valid_results) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot 1: Urbanization scores
    ax1.barh(valid_results['address'], valid_results['urbanization_score'], 
             color=['#2ecc71' if s >= 80 else '#f39c12' if s >= 60 else '#e74c3c' 
                    for s in valid_results['urbanization_score']])
    ax1.set_xlabel('Urbanization Score')
    ax1.set_title('Urbanization Scores by Location')
    ax1.set_xlim(0, 100)
    
    # Plot 2: Distance to nearest Woolworths
    ax2.barh(valid_results['address'], valid_results['distance_km'],
             color=['#3498db' if d < 5 else '#9b59b6' if d < 15 else '#e67e22' 
                    for d in valid_results['distance_km']])
    ax2.set_xlabel('Distance (km)')
    ax2.set_title('Distance to Nearest Woolworths')
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\nSUMMARY STATISTICS")
    print("=" * 60)
    print(f"Average urbanization score: {valid_results['urbanization_score'].mean():.1f}")
    print(f"Average distance to Woolworths: {valid_results['distance_km'].mean():.2f} km")
    print(f"\nUrbanization categories:")
    print(valid_results['urbanization_category'].value_counts())
else:
    print("No valid results to visualize")

No valid results to visualize


---
## Future Work: Housing Price Modeling

This section prepares the data structure for future modeling of housing prices as a function of Woolworths proximity.

In [15]:
# Cell 14: Feature Engineering for Future Modeling

def create_proximity_features(coords: Tuple[float, float], stores_df: pd.DataFrame) -> Dict:
    """
    Create features for housing price modeling based on Woolworths proximity.
    
    These features can be used as predictors in a regression model where
    housing price is the dependent variable.
    
    Args:
        coords: (latitude, longitude) of property
        stores_df: DataFrame with Woolworths stores
    
    Returns:
        Dictionary with engineered features
    """
    
    # Find nearest stores
    nearest_stores = find_nearest_stores(coords, stores_df, n=10)
    
    # Get distance statistics
    stats = get_distance_statistics(coords, stores_df)
    
    # Calculate distances by store type
    store_types = ['Full-line', 'Food', 'Convenience', 'Engen Foodstop']
    type_distances = {}
    
    for store_type in store_types:
        type_stores = stores_df[stores_df['store_type'] == store_type]
        if len(type_stores) > 0:
            distances = type_stores.apply(
                lambda row: calculate_distance(coords, (row['latitude'], row['longitude'])),
                axis=1
            )
            type_distances[f'dist_nearest_{store_type.lower().replace(" ", "_")}'] = distances.min()
            type_distances[f'count_within_5km_{store_type.lower().replace(" ", "_")}'] = (distances < 5).sum()
            type_distances[f'count_within_10km_{store_type.lower().replace(" ", "_")}'] = (distances < 10).sum()
        else:
            type_distances[f'dist_nearest_{store_type.lower().replace(" ", "_")}'] = np.nan
            type_distances[f'count_within_5km_{store_type.lower().replace(" ", "_")}'] = 0
            type_distances[f'count_within_10km_{store_type.lower().replace(" ", "_")}'] = 0
    
    # Urbanization score
    classification = classify_urbanization(nearest_stores.iloc[0]['distance_km'])
    
    # Create feature dictionary
    features = {
        # Basic distance features
        'distance_nearest_woolies': nearest_stores.iloc[0]['distance_km'],
        'distance_2nd_nearest': nearest_stores.iloc[1]['distance_km'] if len(nearest_stores) > 1 else np.nan,
        'distance_3rd_nearest': nearest_stores.iloc[2]['distance_km'] if len(nearest_stores) > 2 else np.nan,
        
        # Aggregate statistics
        'mean_distance_top5': nearest_stores.head(5)['distance_km'].mean(),
        'median_distance_all': stats['median_distance'],
        
        # Store density features
        'count_within_5km': len(nearest_stores[nearest_stores['distance_km'] < 5]),
        'count_within_10km': len(nearest_stores[nearest_stores['distance_km'] < 10]),
        'count_within_20km': len(nearest_stores[nearest_stores['distance_km'] < 20]),
        
        # Urbanization
        'urbanization_score': classification['score'],
        'urbanization_category': classification['category'],
        
        # Location
        'latitude': coords[0],
        'longitude': coords[1],
    }
    
    # Add store-type specific features
    features.update(type_distances)
    
    return features


def create_modeling_dataset(addresses_with_prices: List[Dict], stores_df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a dataset ready for modeling.
    
    Args:
        addresses_with_prices: List of dicts with 'address' and 'price' keys
        stores_df: DataFrame with Woolworths stores
    
    Returns:
        DataFrame with features and target variable
    """
    
    dataset = []
    
    for item in addresses_with_prices:
        address = item['address']
        price = item.get('price', None)
        
        coords = geocode_address(address)
        if coords:
            features = create_proximity_features(coords, stores_df)
            features['address'] = address
            features['price'] = price
            dataset.append(features)
            time.sleep(1)  # Rate limiting
    
    return pd.DataFrame(dataset)


# Example: Create features for a sample location
print("Creating example feature set for modeling:")
print("=" * 60)

sample_coords = (-26.1076, 28.0567)  # Sandton
sample_features = create_proximity_features(sample_coords, woolworths_stores)

print("\nFeatures for sample location (Sandton):")
print("-" * 60)
for feature, value in sample_features.items():
    if isinstance(value, float):
        print(f"{feature:40s}: {value:.2f}")
    else:
        print(f"{feature:40s}: {value}")

print("\n" + "=" * 60)
print("These features can be used as predictors in a regression model")
print("where housing price is the dependent variable.")
print("\nExample usage:")
print("  X = features[['distance_nearest_woolies', 'urbanization_score', ...]]")
print("  y = prices")
print("  model = LinearRegression().fit(X, y)")

Creating example feature set for modeling:

Features for sample location (Sandton):
------------------------------------------------------------
distance_nearest_woolies                : 0.00
distance_2nd_nearest                    : 4.60
distance_3rd_nearest                    : 6.24
mean_distance_top5                      : 14.46
median_distance_all                     : 500.64
count_within_5km                        : 2
count_within_10km                       : 3
count_within_20km                       : 4
urbanization_score                      : 100
urbanization_category                   : Highly Urban
latitude                                : -26.11
longitude                               : 28.06
dist_nearest_full-line                  : 0.00
count_within_5km_full-line              : 1
count_within_10km_full-line             : 1
dist_nearest_food                       : 4.60
count_within_5km_food                   : 1
count_within_10km_food                  : 1
dist_nearest_conv

---
## Summary & Next Steps

### What This Notebook Does:
1. **Data Collection**: Fetches/uses Woolworths store locations across South Africa
2. **Geocoding**: Converts addresses to coordinates using OpenStreetMap
3. **Distance Calculation**: Computes distances to all Woolworths stores
4. **Classification**: Categorizes locations as Urban/Rural based on proximity
5. **Visualization**: Creates interactive maps with Folium
6. **Batch Analysis**: Processes multiple locations simultaneously
7. **Feature Engineering**: Prepares data for future housing price modeling

### Key Functions:
- `geocode_address(address)` - Convert address to coordinates
- `find_nearest_stores(coords, n)` - Find n nearest Woolworths
- `analyze_location(address)` - Complete analysis of an address
- `create_proximity_map(coords)` - Generate interactive map
- `classify_urbanization(distance)` - Classify urban/rural status
- `create_proximity_features(coords)` - Create modeling features

### Next Steps for Enhancement:
1. **Better Store Data**: Replace sample data with complete Woolworths API scraping
2. **Add More Stores**: Expand to include more store locations across SA
3. **Alternative Geocoders**: Add Google Maps API or other geocoding services
4. **Property Price Integration**: Add actual housing price data for modeling
5. **Advanced Features**: Include population density, income levels, etc.
6. **Machine Learning**: Build predictive models for property values
7. **Web Interface**: Create a web app for easy address input
8. **Export Functionality**: Save results to CSV/Excel